In [12]:
import torch                                              # PyTorch main package
import torch.nn as nn                                     # Neural network layers / losses
from torch.utils.data import DataLoader                   # Mini-batch data loading
from torchvision import datasets, transforms, models       # Datasets, transforms, pretrained models
from tqdm import tqdm                                     # Progress bars

ROOT = "."                                                # Current folder; must contain .\MNIST\raw (download=True will create .\MNIST\processed)
BS = 128                                                  # Batch size
EPOCHS = 3                                                # Number of training epochs
LR = 1e-3                                                 # Learning rate
device = "cuda" if torch.cuda.is_available() else "cpu"    # Use GPU if available else CPU

tfm_train = transforms.Compose([                          # Training transforms pipeline (applied to each image)
    transforms.Resize(224),                               # Resize MNIST 28x28 -> 224x224 (MobileNetV3 expects ImageNet-like size)
    transforms.Grayscale(num_output_channels=3),           # Convert 1-channel grayscale -> 3 channels (RGB-like) for pretrained weights
    transforms.RandomRotation(10),                         # Light augmentation: random rotation up to ±10 degrees
    transforms.ToTensor(),                                 # Convert PIL image -> torch tensor in [0,1]
    transforms.Normalize([0.485,0.456,0.406],              # Normalise with ImageNet mean (for pretrained model compatibility)
                         [0.229,0.224,0.225]),             # Normalise with ImageNet std
])

tfm_test = transforms.Compose([                           # Test/validation transforms (no augmentation)
    transforms.Resize(224),                               # Same resizing as training
    transforms.Grayscale(num_output_channels=3),           # Same 1->3 channel conversion
    transforms.ToTensor(),                                 # Same tensor conversion
    transforms.Normalize([0.485,0.456,0.406],              # Same ImageNet normalisation mean
                         [0.229,0.224,0.225]),             # Same ImageNet normalisation std
])

train_ds = datasets.MNIST(root=ROOT,                      # Create training dataset object
                          train=True,                     # Use training split
                          download=True,                  # Build processed/ from raw (and download if raw missing)
                          transform=tfm_train)            # Apply training transforms

test_ds  = datasets.MNIST(root=ROOT,                      # Create test dataset object
                          train=False,                    # Use test split
                          download=True,                  # Ensure processed/ exists
                          transform=tfm_test)             # Apply test transforms

train_dl = DataLoader(train_ds,                           # DataLoader for training
                      batch_size=BS,                      # Use batch size BS
                      shuffle=True,                       # Shuffle training samples each epoch
                      num_workers=2,                      # Worker processes for loading (set 0 if Windows issues)
                      pin_memory=True)                    # Faster host->GPU copies (safe even on CPU)

test_dl  = DataLoader(test_ds,                            # DataLoader for testing
                      batch_size=BS,                      # Use same batch size
                      shuffle=False,                      # Do not shuffle test data
                      num_workers=2,                      # Worker processes
                      pin_memory=True)                    # Pin memory for speed

m = models.mobilenet_v3_large(pretrained=True)            # Load MobileNetV3-Large with ImageNet pretrained weights (torchvision 1.8-style)
m.classifier[-1] = nn.Linear(m.classifier[-1].in_features,# Replace final classifier layer input size
                             10)                          # Output 10 classes for digits 0..9
m = m.to(device)                                          # Move model to GPU/CPU

for p in m.features.parameters():                         # Iterate over all backbone (feature extractor) parameters
    p.requires_grad = False                               # Freeze them: train only the classifier first (faster + stable)

opt = torch.optim.Adam(filter(lambda p: p.requires_grad,  # Create Adam optimiser over trainable parameters only
                              m.parameters()),
                       lr=LR)                             # Set learning rate
crit = nn.CrossEntropyLoss()                              # Classification loss for logits vs integer labels

def evaluate():                                           # Define evaluation function on test set
    m.eval()                                              # Put model in evaluation mode (disables dropout, uses running stats)
    correct = 0                                           # Count of correct predictions
    total = 0                                             # Count of total samples
    loss_sum = 0.0                                        # Sum of losses over all samples
    with torch.no_grad():                                 # Disable gradients (faster + less memory)
        for x, y in test_dl:                              # Loop over test batches
            x = x.to(device)                              # Move images to device
            y = y.to(device)                              # Move labels to device
            out = m(x)                                    # Forward pass -> logits
            loss = crit(out, y)                           # Compute loss for this batch
            loss_sum += loss.item() * y.size(0)           # Accumulate total loss (weighted by batch size)
            pred = out.argmax(1)                          # Predicted class = index of max logit
            correct += (pred == y).sum().item()           # Add number of correct predictions
            total += y.numel()                            # Add number of labels in this batch
    return loss_sum / total, correct / total              # Return average loss and accuracy


In [13]:

for epoch in range(1, EPOCHS + 1):                        # Loop over epochs (1..EPOCHS)
    m.train()                                             # Put model in training mode
    pbar = tqdm(train_dl, desc=f"Train {epoch}/{EPOCHS}") # Progress bar over training batches
    running = 0.0                                         # Running sum of batch losses (for display)
    for x, y in pbar:                                     # Loop over training batches
        x = x.to(device)                                  # Move images to device
        y = y.to(device)                                  # Move labels to device
        opt.zero_grad()                                   # Clear old gradients
        out = m(x)                                        # Forward pass -> logits
        loss = crit(out, y)                               # Compute loss
        loss.backward()                                   # Backpropagate to compute gradients
        opt.step()                                        # Update trainable parameters
        running += loss.item()                            # Accumulate loss for progress display
        pbar.set_postfix(loss=running / len(pbar))        # Show average loss so far in tqdm

    val_loss, val_acc = evaluate()                        # Evaluate after each epoch
    print(f"Epoch {epoch}: val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")  # Print metrics

torch.save(m.state_dict(), "mobilenetv3_mnist.pth")        # Save trained weights to a file
print("Saved: mobilenetv3_mnist.pth")                      # Confirm save path

# If you want full fine-tuning after this:
# 1) unfreeze: for p in m.features.parameters(): p.requires_grad = True
# 2) re-create optimiser with smaller LR, e.g. opt = torch.optim.Adam(m.parameters(), lr=1e-4)

Train 1/3: 100%|██████████| 469/469 [01:14<00:00,  6.28it/s, loss=0.3]  


Epoch 1: val_loss=0.7466  val_acc=0.7672


Train 2/3: 100%|██████████| 469/469 [01:21<00:00,  5.78it/s, loss=0.162] 


Epoch 2: val_loss=0.1682  val_acc=0.9485


Train 3/3: 100%|██████████| 469/469 [01:18<00:00,  5.97it/s, loss=0.138] 


Epoch 3: val_loss=0.1386  val_acc=0.9556
Saved: mobilenetv3_mnist.pth
